# Project Directory Notes (Concise)

## 1) Root-level files
- **`app.py`**: Main Dash entrypoint; wires route-based screen rendering (`/setup`, `/train`, `/compare`, `/dashboard`, `/drilldown`, `/logs`, `/settings`) and defaults to setup unless trained model exists.
- **`requirements.txt` / `environment.yml`**: Environment definitions (Python stack for Dash + ML + deep learning + notebooks).
- **`Exploratory Data Analysis.ipynb` / `Machine Learning.ipynb` / `JULLIANE-NB.ipynb`**: Notebook workflow files for analysis, training experimentation, and documentation.
- **`TODO.txt`**: Task tracking / project reminders.

## 2) App UI layer (`app/`)
- **`app/__init__.py`**: Creates single Dash app/server instance.
- **`app/screens/*.py`**: Page-level layouts (setup, training, model comparison, dashboard, drilldown, logs, settings).
- **`app/components/*.py`**: Reusable UI widgets (line chart, upload form, navbar, datatable wrapper).
- **`app/callbacks/initial_setup_callbacks.py`**: Multi-step setup callback logic (file uploads, state progression, clearing uploaded files).
- **`app/assets/`**: CSS + icons for styling and branding.

## 3) Data ingestion (`DataLoaders/`)
- **`revenues.py` (`Revenues`)**: Async loads Excel revenue ledgers, cleans columns/types, updates due dates from settings sheet, computes receivables, pseudonymizes IDs, standardizes account names.
- **`enrollees.py` (`Enrollees`)**: Derives enrollee-level table from revenues, maps plan/grade/education levels, tags refunded students.
- **`bad_debts.py` (`BadDebtsExpense`)**: Builds synthetic bad-debt expense records from aged receivables (older unpaid balances).
- **`settings.py` (`Settings`)**: Reads `settings.json` and resolves install/database paths plus runtime flags.

## 4) Feature engineering (`FeatureEngineering/`)
- **`credit_sales_eda.py` (`CreditSales`)**: Core receivables-to-credit-sales transformation for analysis; handles single vs multiple due-date logic, discounts/adjustments, payment aging buckets, net receivables.
- **`credit_sales_machine_learning.py` (`CreditSales`)**: ML-oriented variant adding model-ready features (e.g., payment timing / full-payment date and encoded predictors).
- **`consecutive_years.py`**: Computes per-student consecutive enrollment streaks (excluding refunded rows from streak counting).
- **`days_sales_outstanding.py` (`DSO`)**: Computes period DSO + rolling 12-month DSO metrics.
- **`_ARCHIVED_*` files**: Older / experimental implementations kept for reference.

## 5) Analysis helpers (`Analysis/`)
- **`enrollment_statistics.py`**: Produces yearly enrollment summaries (counts or percentages), including new enrollees and consecutive-year breakdowns.

## 6) Machine learning package (`MachineLearning/`)
- **`Models/base_pipeline.py`**: Shared abstract training/evaluation scaffold.
- **Classical model pipelines**: `ada_boost.py`, `decision_tree.py`, `gaussian_naive_bayes.py`, `k_nearest_neighbor.py`, `random_forest.py`, `xg_boost.py`.
- **Neural model pipelines**: `multi_layer_perceptron.py` (PyTorch MLP + optional SHAP feature selection), `recurrent_neural_network.py` (Keras RNN), `transformer.py` (Hugging Face text classifier).
- **`parameters.json`**: Hyperparameter search/config source per model.

### ML utilities (`MachineLearning/Utils/`)
- **`data_partitioning.py`**: Date-aware train/test splitting.
- **`data_preparation.py`**: End-to-end prep (label encoding, balancing via SMOTE-family/hybrid, scaling).
- **`hybrid_balance.py`**: Custom balancing strategy.
- **`generate_survival_features.py` / `adjust_survival_time_periods.py`**: Survival-analysis-derived features and target time adjustments.
- **`generate_thresholds.py`**: Threshold grid generation.
- **`load_parameters.py`**: Parameter loader from JSON.
- **`data_evaluation.py`**: Core metrics aggregation (accuracy/precision/recall/F1/AUC).
- **`run_models_parallel.py` / `run_models_single_threaded.py`**: Experiment orchestration (baseline vs enhanced survival-feature runs), including export to results files.
- **`model_manager.py`**: Simple trained-model existence check for UI routing.

## 7) Utility + storage folders
- **`Utils/Pseudonymizer/pseudonymizer.py`**: ID anonymization with persistent JSON cache (`Database/pseudonym_cache.json`).
- **`Database/`**: Data storage (cache and other database assets).
- **`Images/`**: Visual assets used in reports/notebooks/UI.

## 8) Overall project meaning (high-level)
This repo is an **end-to-end tuition receivables intelligence system**: it ingests school finance ledgers, engineers receivable/payment behavior features, trains and compares multiple ML models (including survival-enhanced variants), and surfaces outputs in a Dash dashboard for setup, monitoring, drilldown, and audit-style review.

## 2) Data Description: Tables and Variables

### Core Tables (logical datasets)
1. **Revenues (raw/cleaned ledger)**
   - Built by `DataLoaders/revenues.py`
   - Grain: transaction-level school finance row
   - Main columns after standardization:
     - `entry_date`, `due_date`, `school_year`
     - `student_id_pseudonimized`
     - `category_name`, `discount_refund_applied_to`
     - `amount_due`, `amount_paid`, `receivables`
     - `account_name`

2. **Enrollees (derived student-year enrollment table)**
   - Built by `DataLoaders/enrollees.py`
   - Grain: student + school_year (+ plan category)
   - Main columns:
     - `school_year`, `student_id_pseudonimized`
     - `grade_level`, `education_level`
     - `plan_type`, `enrollment_date`
     - `has_refunded`

3. **Credit Sales (EDA variant)**
   - Built by `FeatureEngineering/credit_sales_eda.py`
   - Grain: student + school_year + category + due date bucket summary
   - Main columns include:
     - `credit_sale_amount`, `adjusted_credit_amount`, `net_receivables`
     - payment aging buckets (`prepayments`, `30_days`, ..., `180_above`)

4. **Credit Sales (ML variant)**
   - Built by `FeatureEngineering/credit_sales_machine_learning.py`
   - Grain: model-ready invoice/credit-sales record
   - Adds/derives modeling features such as payment timing and date-based features.

5. **Enrollment Statistics / DSO / Bad Debts outputs**
   - `Analysis/enrollment_statistics.py` → yearly enrollment and streak summaries
   - `FeatureEngineering/days_sales_outstanding.py` → DSO time-series metrics
   - `DataLoaders/bad_debts.py` → bad debt expense adjustment rows

### Important Variable Families
- **Identifiers**: `student_id_pseudonimized`, `school_year`, `category_name`
- **Dates**: `entry_date`, `due_date`, `payment_date`, `date_fully_paid`
- **Amounts**: `amount_due`, `amount_paid`, `receivables`, `credit_sale_amount`
- **Labels/Targets**: late-payment related target configured in ML args (via `target_feature`)
- **Audit flags**: `has_refunded`, transaction category/refund/discount mappings

In [7]:
# Optional quick data dictionary snapshot (manual schema map)
import pandas as pd

schema_rows = [
    ("revenues", "entry_date", "datetime", "Transaction entry date"),
    ("revenues", "due_date", "datetime", "Expected due date"),
    ("revenues", "school_year", "int", "Academic year"),
    ("revenues", "student_id_pseudonimized", "str", "Pseudonymized student ID"),
    ("revenues", "category_name", "str", "Charge/payment category"),
    ("revenues", "amount_due", "float", "Billed amount"),
    ("revenues", "amount_paid", "float", "Paid amount"),
    ("revenues", "receivables", "float", "Outstanding receivable"),
    ("enrollees", "plan_type", "str", "Payment plan type"),
    ("enrollees", "grade_level", "str", "Grade code"),
    ("enrollees", "education_level", "str", "Mapped level band"),
    ("credit_sales", "credit_sale_amount", "float", "Computed credit sales"),
    ("credit_sales", "adjusted_credit_amount", "float", "Credit sales net of prepayments"),
    ("credit_sales", "net_receivables", "float", "Remaining balance after allocations"),
]

df_schema = pd.DataFrame(schema_rows, columns=["table", "variable", "dtype", "meaning"])
display(df_schema)


,table,variable,dtype,meaning
0,revenues,entry_date,datetime,Transaction entry date
1,revenues,due_date,datetime,Expected due date
2,revenues,school_year,int,Academic year
3,revenues,student_id_pseudonimized,str,Pseudonymized student ID
4,revenues,category_name,str,Charge/payment category
5,revenues,amount_due,float,Billed amount
6,revenues,amount_paid,float,Paid amount
7,revenues,receivables,float,Outstanding receivable
8,enrollees,plan_type,str,Payment plan type
9,enrollees,grade_level,str,Grade code


## 3) Processing Pipeline (End-to-End)

### A. Ingestion and Standardization
1. Read ledger Excel files (`DataLoaders/revenues.py`)
2. Clean/standardize data types and key columns
3. Adjust due dates using settings sheet
4. Pseudonymize student IDs (`Utils/Pseudonymizer/pseudonymizer.py`)

### B. Derived Analytical Tables
1. Build enrollment-level table (`DataLoaders/enrollees.py`)
2. Build credit sales and receivables tables (`FeatureEngineering/credit_sales_eda.py` and `credit_sales_machine_learning.py`)
3. Compute supporting analytics:
   - consecutive enrollment years (`consecutive_years.py`)
   - DSO (`days_sales_outstanding.py`)
   - bad debts (`bad_debts.py`)

### C. Modeling Preparation
1. Build training/test split using due-date aware partitioning (`MachineLearning/Utils/data_partitioning.py`)
2. Encode labels, balance classes (SMOTE / hybrid), scale features (`data_preparation.py`)
3. Optionally generate survival-based feature augmentations (`generate_survival_features.py`)

### D. Model Training and Evaluation
1. Run selected models (`MachineLearning/Models/*`)
2. Evaluate metrics using `MachineLearning/Utils/data_evaluation.py`
3. Save combined experiment outputs (baseline vs enhanced feature sets)

### E. Application / Reporting Layer
1. Serve flows in Dash app screens (`app/screens/*`)
2. Use setup/training/comparison/dashboard screens for operations and monitoring

## 4) Machine Learning Models and Results

### Models in Scope
- **Classical models**: AdaBoost, Decision Tree, Gaussian Naive Bayes, KNN, Random Forest, XGBoost
- **Neural models**: Multi-layer Perceptron (PyTorch), Recurrent Neural Network (Keras), Transformer (Hugging Face)

### Experiment Design
- Compare **baseline** features vs **enhanced** features (survival-augmented)
- Run across balancing strategies and thresholds
- Aggregate metrics:
  - Accuracy
  - Precision (macro)
  - Recall (macro)
  - F1 (macro)
  - ROC-AUC (macro, where available)

### Expected Result Artifacts
- Typical output path in code: `MachineLearning/Results/model_results.xlsx`
- Includes model name, parameters, strategy, threshold, baseline metrics, enhanced metrics, selected features

In [8]:
# Results loader and concise summary table
from pathlib import Path
import pandas as pd

results_path = Path("MachineLearning/Results/model_results.xlsx")

if results_path.exists():
    df_results = pd.read_excel(results_path)
    print(f"Loaded results: {results_path} ({len(df_results)} rows)")

    candidate_metric_cols = [
        "baseline_f1_macro", "enhanced_f1_macro",
        "baseline_accuracy", "enhanced_accuracy",
        "baseline_roc_auc_macro", "enhanced_roc_auc_macro"
    ]
    existing_metrics = [c for c in candidate_metric_cols if c in df_results.columns]

    core_cols = [c for c in ["model", "parameters", "balance_strategy", "undersample_threshold"] if c in df_results.columns]
    display_cols = core_cols + existing_metrics

    if display_cols:
        display(df_results[display_cols].head(20))

    if "enhanced_f1_macro" in df_results.columns:
        best_row = df_results.sort_values("enhanced_f1_macro", ascending=False).head(1)
        print("Best run by enhanced_f1_macro:")
        display(best_row[display_cols if display_cols else best_row.columns])
    else:
        print("No 'enhanced_f1_macro' column found; review available metrics above.")
else:
    print(f"Results file not found: {results_path}")
    print("Run the model experiment pipeline first, then re-run this cell.")


Results file not found: MachineLearning\Results\model_results.xlsx
Run the model experiment pipeline first, then re-run this cell.


## 5) Double-Check Section: Process and Code Validation

Use this section to verify that reported results are reproducible and consistent.

### Validation Checklist
1. **Environment check**: Python version and key package versions match expected setup.
2. **Input data check**: required columns exist and date/numeric fields are parseable.
3. **Pipeline consistency**: no missing critical intermediate outputs before model run.
4. **Result consistency**:
   - metrics are in valid range [0, 1] where applicable
   - best-model selection logic is deterministic
5. **Code/process integrity**:
   - model/result artifacts loaded from expected path
   - no silent file-not-found conditions
   - assumptions and thresholds explicitly documented

> Recommendation: Run this section whenever code, parameters, data inputs, or environment changes.

In [9]:
# Lightweight verification checks for process + results
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.version.split()[0])

results_path = Path("MachineLearning/Results/model_results.xlsx")
if not results_path.exists():
    print("[WARN] Results file missing:", results_path)
else:
    df = pd.read_excel(results_path)
    print("[OK] Results rows:", len(df))

    bounded_metric_candidates = [
        "baseline_accuracy", "enhanced_accuracy",
        "baseline_precision_macro", "enhanced_precision_macro",
        "baseline_recall_macro", "enhanced_recall_macro",
        "baseline_f1_macro", "enhanced_f1_macro",
        "baseline_roc_auc_macro", "enhanced_roc_auc_macro",
    ]
    bounded_metrics = [c for c in bounded_metric_candidates if c in df.columns]

    out_of_range_report = {}
    for col in bounded_metrics:
        series = pd.to_numeric(df[col], errors="coerce").dropna()
        invalid = series[(series < 0) | (series > 1)]
        if len(invalid) > 0:
            out_of_range_report[col] = len(invalid)

    if out_of_range_report:
        print("[WARN] Some metric values are outside [0,1]:", out_of_range_report)
    else:
        print("[OK] Metric ranges look valid for available bounded metrics.")

    required_context_cols = ["model", "parameters", "balance_strategy"]
    missing_context = [c for c in required_context_cols if c not in df.columns]
    if missing_context:
        print("[WARN] Missing context columns:", missing_context)
    else:
        print("[OK] Core experiment context columns are present.")

print("Verification cell complete.")


Python: 3.13.5
[WARN] Results file missing: MachineLearning\Results\model_results.xlsx
Verification cell complete.


## 6) Consolidated Graphics, Interpretations, RNN Diagnosis, and Run Counting

This section does four things:
1. Generates the key model-result graphics from experiment output files.
2. Prints concise interpretation notes from the computed metrics.
3. Explains why RNN did not run reliably with the current data/pipeline interface.
4. Audits total run counts (to verify claims like “~700 models”).

> The code is defensive: if no results file exists yet, it will report what is missing and still run the run-count/RNN diagnostics.

In [10]:
# Graphics + interpretation generator for model runs
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- locate results file ----
results_candidates = [
    Path("MachineLearning/Results/model_results.xlsx"),
    Path("MachineLearning/Results/model_results.csv"),
]

results_path = next((p for p in results_candidates if p.exists()), None)
if results_path is None:
    print("[INFO] No results file found in MachineLearning/Results.")
    print("       Expected one of:")
    for p in results_candidates:
        print(f"       - {p}")
else:
    print(f"[OK] Using results file: {results_path}")
    df = pd.read_csv(results_path) if results_path.suffix.lower() == ".csv" else pd.read_excel(results_path)
    print(f"[OK] Rows loaded: {len(df)}")

    # ---- choose primary ranking metric ----
    metric_priority = ["enhanced_f1_macro", "enhanced_roc_auc_macro", "enhanced_accuracy"]
    primary_metric = next((m for m in metric_priority if m in df.columns), None)

    if primary_metric is None:
        print("[WARN] No expected enhanced metric found. Available columns:")
        print(df.columns.tolist())
    else:
        print(f"[OK] Primary metric: {primary_metric}")

        # --- ensure useful columns ---
        if "balance_strategy_full" not in df.columns and "balance_strategy" in df.columns:
            if "undersample_threshold" in df.columns:
                df["balance_strategy_full"] = df.apply(
                    lambda r: f"{r['balance_strategy']}_{r['undersample_threshold']}" if r.get("balance_strategy") == "hybrid" else r.get("balance_strategy"),
                    axis=1
                )
            else:
                df["balance_strategy_full"] = df["balance_strategy"]

        # -------------------- Plot 1: Top configurations --------------------
        top_n = min(12, len(df))
        top_df = df.sort_values(primary_metric, ascending=False).head(top_n).copy()
        top_df["label"] = top_df.apply(
            lambda r: f"{r.get('model','?')} | {r.get('balance_strategy','?')}", axis=1
        )

        plt.figure(figsize=(12, max(5, top_n * 0.4)))
        plt.barh(top_df["label"][::-1], top_df[primary_metric][::-1])
        plt.title(f"Top {top_n} Runs by {primary_metric}")
        plt.xlabel(primary_metric)
        plt.tight_layout()
        plt.show()

        # -------------------- Plot 2: Baseline vs Enhanced delta by model --------------------
        baseline_metric = primary_metric.replace("enhanced_", "baseline_")
        if baseline_metric in df.columns and "model" in df.columns:
            delta_df = df[["model", baseline_metric, primary_metric]].copy()
            delta_df["delta_enhanced_minus_baseline"] = delta_df[primary_metric] - delta_df[baseline_metric]
            model_delta = delta_df.groupby("model")["delta_enhanced_minus_baseline"].mean().sort_values(ascending=False)

            plt.figure(figsize=(10, 5))
            plt.bar(model_delta.index, model_delta.values)
            plt.axhline(0, linestyle="--")
            plt.title(f"Mean Gain from Enhanced Features ({primary_metric} - {baseline_metric})")
            plt.ylabel("Mean Delta")
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            plt.show()
        else:
            print(f"[INFO] Skipping delta plot; missing '{baseline_metric}' or 'model'.")

        # -------------------- Plot 3: Balance strategy comparison --------------------
        if "balance_strategy_full" in df.columns:
            strat = df.groupby("balance_strategy_full")[primary_metric].mean().sort_values(ascending=False)
            plt.figure(figsize=(10, 5))
            plt.bar(strat.index, strat.values)
            plt.title(f"Average {primary_metric} by Balance Strategy")
            plt.ylabel(f"Mean {primary_metric}")
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            plt.show()

        # -------------------- Plot 4: Heatmap-like pivot (table view) --------------------
        if "model" in df.columns and "balance_strategy_full" in df.columns:
            pivot = df.pivot_table(index="model", columns="balance_strategy_full", values=primary_metric, aggfunc="mean")
            display(pivot.style.background_gradient(cmap="YlGnBu"))

        # -------------------- Concise interpretation block --------------------
        best_row = df.sort_values(primary_metric, ascending=False).iloc[0]
        print("\nINTERPRETATION NOTES")
        print("- Best single run:")
        print(f"  model={best_row.get('model')}, strategy={best_row.get('balance_strategy')}, {primary_metric}={best_row.get(primary_metric):.4f}")

        if baseline_metric in df.columns:
            mean_delta = (df[primary_metric] - df[baseline_metric]).mean()
            direction = "improved" if mean_delta > 0 else ("declined" if mean_delta < 0 else "stayed equal")
            print(f"- On average, enhanced features {direction} performance by {mean_delta:.4f} ({primary_metric}).")

        if "balance_strategy_full" in df.columns:
            best_strat = df.groupby("balance_strategy_full")[primary_metric].mean().sort_values(ascending=False).index[0]
            print(f"- Best average balancing strategy: {best_strat}.")

        if "model" in df.columns:
            best_model = df.groupby("model")[primary_metric].mean().sort_values(ascending=False).index[0]
            print(f"- Best average model family: {best_model}.")


[INFO] No results file found in MachineLearning/Results.
       Expected one of:
       - MachineLearning\Results\model_results.xlsx
       - MachineLearning\Results\model_results.csv


In [11]:
# Run-count audit: verify whether "~700 models" is accurate
import json
from pathlib import Path

params_path = Path("MachineLearning/parameters.json")
if not params_path.exists():
    print("[WARN] Missing MachineLearning/parameters.json")
else:
    params = json.loads(params_path.read_text(encoding="utf-8"))
    param_counts = {k: len(v) for k, v in params.items() if isinstance(v, list)}

    # Current active setup from Machine Learning.ipynb (based on your latest notebook code)
    active_models_current = [
        "ada_boost", "decision_tree", "gaussian_naive_bayes", "knn", "random_forest", "xgboost"
    ]
    active_strategies_current = ["borderline_smote", "smoteenn", "smotetomek"]
    hybrid_thresholds_current = []  # hybrid not active in current setup

    total_param_sets_current = sum(param_counts.get(m, 0) for m in active_models_current)
    strategy_units_current = len(active_strategies_current) + len(hybrid_thresholds_current)

    # In run_models_parallel.py each row stores one (model, param, strategy/threshold) but contains baseline+enhanced metrics
    result_rows_current = total_param_sets_current * strategy_units_current
    model_fits_current = result_rows_current * 2  # baseline + enhanced

    print("CURRENT NOTEBOOK CONFIG")
    print("- Parameter sets across active models:", total_param_sets_current)
    print("- Strategy units:", strategy_units_current)
    print("- Expected result rows:", result_rows_current)
    print("- Expected model fits (baseline + enhanced):", model_fits_current)

    # Scenario often discussed: 5 strategies including hybrid with thresholds 0.5..0.9
    strategies_full = ["smote", "borderline_smote", "smoteenn", "smotetomek", "hybrid"]
    hybrid_thresholds_full = [0.5, 0.6, 0.7, 0.8, 0.9]
    strategy_units_full = 4 + len(hybrid_thresholds_full)  # non-hybrid(4) + hybrid thresholds

    result_rows_full = total_param_sets_current * strategy_units_full
    model_fits_full = result_rows_full * 2

    print("\nIF FULL STRATEGIES WERE USED (smote + borderline + smoteenn + smotetomek + hybrid[0.5..0.9])")
    print("- Strategy units:", strategy_units_full)
    print("- Expected result rows:", result_rows_full)
    print("- Expected model fits:", model_fits_full)

    if result_rows_current < 700 <= model_fits_full:
        print("\nInterpretation: '700 models' may be a mixed counting convention.")
        print("- Counting result rows (unique model+param+strategy): usually below 700 here.")
        print("- Counting baseline and enhanced fits separately can exceed 700 under broader strategy settings.")


CURRENT NOTEBOOK CONFIG
- Parameter sets across active models: 57
- Strategy units: 3
- Expected result rows: 171
- Expected model fits (baseline + enhanced): 342

IF FULL STRATEGIES WERE USED (smote + borderline + smoteenn + smotetomek + hybrid[0.5..0.9])
- Strategy units: 9
- Expected result rows: 513
- Expected model fits: 1026

Interpretation: '700 models' may be a mixed counting convention.
- Counting result rows (unique model+param+strategy): usually below 700 here.
- Counting baseline and enhanced fits separately can exceed 700 under broader strategy settings.


In [12]:
# RNN compatibility diagnosis (exact mismatch checks)
from pathlib import Path
import json

rnn_file = Path("MachineLearning/Models/recurrent_neural_network.py")
params_file = Path("MachineLearning/parameters.json")
runner_file = Path("MachineLearning/Utils/run_models_parallel.py")

def contains(text, needle):
    return needle in text

if not (rnn_file.exists() and params_file.exists() and runner_file.exists()):
    print("[WARN] Missing one or more required files for diagnosis.")
else:
    rnn_src = rnn_file.read_text(encoding="utf-8")
    runner_src = runner_file.read_text(encoding="utf-8")
    params = json.loads(params_file.read_text(encoding="utf-8"))
    nn_rnn_params = params.get("nn_rnn", [])

    first_param_keys = set(nn_rnn_params[0].keys()) if nn_rnn_params else set()
    expected_rnn_keys = {"units", "activation", "optimizer", "loss", "metrics", "input_shape"}

    print("RNN DIAGNOSIS")

    # 1) Runner API contract mismatch
    needs_initialize = contains(runner_src, ".initialize_model()")
    needs_fit = contains(runner_src, ".fit(")
    needs_evaluate = contains(runner_src, ".evaluate()")

    has_build = contains(rnn_src, "def build_model(")
    has_train = contains(rnn_src, "def train(")
    has_evaluation = contains(rnn_src, "def evaluation(")

    print("1) Runner-method compatibility")
    print(f"   - Runner expects initialize_model/fit/evaluate: {needs_initialize and needs_fit and needs_evaluate}")
    print(f"   - RNN provides build_model/train/evaluation: {has_build and has_train and has_evaluation}")

    # 2) Parameter schema mismatch
    missing_in_json = expected_rnn_keys - first_param_keys
    extra_in_json = first_param_keys - expected_rnn_keys

    print("2) Parameter schema compatibility")
    print(f"   - JSON nn_rnn keys: {sorted(first_param_keys)}")
    print(f"   - Keys expected by current RNN code: {sorted(expected_rnn_keys)}")
    print(f"   - Missing required keys (for current RNN code): {sorted(missing_in_json)}")
    print(f"   - Unused keys from JSON (in current RNN code): {sorted(extra_in_json)}")

    # 3) Task/output compatibility signals
    uses_binary_output = contains(rnn_src, "Dense(1") and contains(rnn_src, "binary_crossentropy")
    print("3) Output/loss setup")
    print(f"   - RNN hard-coded for binary classification: {uses_binary_output}")
    print("   - Your broader pipeline reports macro multi-class metrics, so label/output shape likely mismatched.")

    # 4) Likely input-shape issue
    requires_input_shape = contains(rnn_src, "input_shape")
    print("4) Input tensor shape")
    print(f"   - RNN requires explicit input_shape (3D sequence format): {requires_input_shape}")
    print("   - Current tabular pipeline prepares 2D feature matrices by default.")

    print("\nConclusion:")
    print("RNN did not run mainly due to interface mismatch with the experiment runner,")
    print("parameter-schema mismatch, and likely data-shape / objective mismatch (sequence-binary vs tabular-multiclass).")


RNN DIAGNOSIS
1) Runner-method compatibility
   - Runner expects initialize_model/fit/evaluate: True
   - RNN provides build_model/train/evaluation: True
2) Parameter schema compatibility
   - JSON nn_rnn keys: ['hidden_units', 'num_layers', 'rnn_type']
   - Keys expected by current RNN code: ['activation', 'input_shape', 'loss', 'metrics', 'optimizer', 'units']
   - Missing required keys (for current RNN code): ['activation', 'input_shape', 'loss', 'metrics', 'optimizer', 'units']
   - Unused keys from JSON (in current RNN code): ['hidden_units', 'num_layers', 'rnn_type']
3) Output/loss setup
   - RNN hard-coded for binary classification: True
   - Your broader pipeline reports macro multi-class metrics, so label/output shape likely mismatched.
4) Input tensor shape
   - RNN requires explicit input_shape (3D sequence format): True
   - Current tabular pipeline prepares 2D feature matrices by default.

Conclusion:
RNN did not run mainly due to interface mismatch with the experiment run

## 7) Thesis-Ready Narrative Summary (Auto-generated)

This section generates a clean narrative paragraph you can directly adapt into Chapter 4 (Results and Discussion). It summarizes:
- best-performing model and strategy,
- enhancement gain (baseline vs enhanced),
- balancing strategy trend,
- run-count interpretation,
- and RNN compatibility findings.

In [13]:
# Auto-generate thesis-ready narrative paragraph
from pathlib import Path
import json
import pandas as pd

paragraphs = []

# 1) Results summary paragraph (if results file exists)
results_candidates = [
    Path("MachineLearning/Results/model_results.xlsx"),
    Path("MachineLearning/Results/model_results.csv"),
]
results_path = next((p for p in results_candidates if p.exists()), None)

if results_path is not None:
    df = pd.read_csv(results_path) if results_path.suffix.lower() == ".csv" else pd.read_excel(results_path)

    metric_priority = ["enhanced_f1_macro", "enhanced_roc_auc_macro", "enhanced_accuracy"]
    primary_metric = next((m for m in metric_priority if m in df.columns), None)

    if primary_metric is not None and len(df) > 0:
        best_row = df.sort_values(primary_metric, ascending=False).iloc[0]
        baseline_metric = primary_metric.replace("enhanced_", "baseline_")

        gain_text = ""
        if baseline_metric in df.columns:
            mean_delta = (df[primary_metric] - df[baseline_metric]).mean()
            gain_text = (
                f"Across all runs, feature enhancement changed {primary_metric} by an average of "
                f"{mean_delta:.4f} (enhanced minus baseline). "
            )

        strat_text = ""
        if "balance_strategy" in df.columns:
            best_strat = df.groupby("balance_strategy")[primary_metric].mean().sort_values(ascending=False).index[0]
            strat_text = f"At aggregate level, {best_strat} produced the strongest mean {primary_metric}. "

        p_results = (
            f"Based on the consolidated experiment output ({len(df)} runs), the highest-performing configuration was "
            f"{best_row.get('model', 'N/A')} under {best_row.get('balance_strategy', 'N/A')} balancing, with "
            f"{primary_metric} = {float(best_row.get(primary_metric)):.4f}. "
            f"{gain_text}{strat_text}"
            f"These findings indicate that model ranking should be interpreted jointly with balancing strategy and feature set design, "
            f"rather than by model family alone."
        )
        paragraphs.append(("Results narrative", p_results))
    else:
        paragraphs.append(("Results narrative", "Results file exists, but expected enhanced metrics were not found."))
else:
    paragraphs.append(("Results narrative", "No results file was found in MachineLearning/Results, so quantitative result narrative could not be generated yet."))

# 2) Run-count narrative (from parameters + known strategy scenarios)
params_path = Path("MachineLearning/parameters.json")
if params_path.exists():
    params = json.loads(params_path.read_text(encoding="utf-8"))
    param_counts = {k: len(v) for k, v in params.items() if isinstance(v, list)}

    active_models_current = [
        "ada_boost", "decision_tree", "gaussian_naive_bayes", "knn", "random_forest", "xgboost"
    ]

    total_param_sets_current = sum(param_counts.get(m, 0) for m in active_models_current)
    strategy_units_current = 3  # borderline_smote, smoteenn, smotetomek
    result_rows_current = total_param_sets_current * strategy_units_current
    model_fits_current = result_rows_current * 2  # baseline + enhanced

    strategy_units_full = 4 + 5  # smote, borderline_smote, smoteenn, smotetomek + hybrid thresholds 0.5..0.9
    result_rows_full = total_param_sets_current * strategy_units_full
    model_fits_full = result_rows_full * 2

    p_count = (
        f"The run-count audit shows that the current active setup corresponds to {result_rows_current} experiment rows "
        f"({model_fits_current} model fits when baseline and enhanced are counted separately). "
        f"Under the broader five-strategy scenario with hybrid thresholds, this scales to {result_rows_full} rows "
        f"({model_fits_full} fits). Therefore, claims such as ‘around 700 models’ are plausible only under a fit-level counting convention, "
        f"not necessarily as unique model-parameter-strategy rows."
    )
    paragraphs.append(("Run-count narrative", p_count))

# 3) RNN narrative (root-cause explanation)
rnn_file = Path("MachineLearning/Models/recurrent_neural_network.py")
runner_file = Path("MachineLearning/Utils/run_models_parallel.py")
if rnn_file.exists() and runner_file.exists():
    rnn_src = rnn_file.read_text(encoding="utf-8")
    runner_src = runner_file.read_text(encoding="utf-8")

    runner_contract = all(k in runner_src for k in [".initialize_model()", ".fit(", ".evaluate()"])
    rnn_contract = all(k in rnn_src for k in ["def build_model(", "def train(", "def evaluation("])
    binary_signals = ("Dense(1" in rnn_src) and ("binary_crossentropy" in rnn_src)

    p_rnn = (
        "The RNN pipeline was not executed in the main experiment loop due to structural incompatibilities: "
        "the experiment runner expects an initialize_model/fit/evaluate interface, while the current RNN class implements "
        "build_model/train/evaluation; additionally, the current RNN implementation is configured as a binary classifier and requires explicit "
        "sequence-style input shape, whereas the shared training pipeline is tabular and evaluated primarily with macro multi-class metrics. "
        f"(Runner contract detected: {runner_contract}; RNN method pattern detected: {rnn_contract}; binary-output signals detected: {binary_signals})."
    )
    paragraphs.append(("RNN narrative", p_rnn))

# Display and printable version
for title, text in paragraphs:
    print(f"\n{title}:\n{text}\n")

full_narrative = " ".join(text for _, text in paragraphs)
print("THESIS-READY COMBINED PARAGRAPH:\n")
print(full_narrative)



Results narrative:
No results file was found in MachineLearning/Results, so quantitative result narrative could not be generated yet.


Run-count narrative:
The run-count audit shows that the current active setup corresponds to 171 experiment rows (342 model fits when baseline and enhanced are counted separately). Under the broader five-strategy scenario with hybrid thresholds, this scales to 513 rows (1026 fits). Therefore, claims such as ‘around 700 models’ are plausible only under a fit-level counting convention, not necessarily as unique model-parameter-strategy rows.


RNN narrative:
The RNN pipeline was not executed in the main experiment loop due to structural incompatibilities: the experiment runner expects an initialize_model/fit/evaluate interface, while the current RNN class implements build_model/train/evaluation; additionally, the current RNN implementation is configured as a binary classifier and requires explicit sequence-style input shape, whereas the shared training p

## 8) Quick Run — Generate Results in ≤10 Minutes

This section runs a **reduced experiment** designed to complete in under 10 minutes, producing a valid `model_results.xlsx` that all earlier sections can read.

### What is cut to hit the time target

| Setting | Full run | Quick run | Reason |
|---------|----------|-----------|--------|
| Balance strategies | 3 (`borderline_smote`, `smoteenn`, `smotetomek`) | **1** (`borderline_smote`) | Eliminates 2× data-prep + survival-feature gen |
| Parameter sets per model | Full grid (4–17 per model) | **1** (first param set only) | Cuts fits 4–17× per model |
| Feature selection | ON (10 permutation repeats) | **OFF** | Single biggest time saver — skips permutation importance shuffling |
| Models | All 6 classical | **3 fastest** (GNB, DT, KNN) | Avoids RF/XGB/AdaBoost tree-depth cost |

With 1 strategy × 1 param × 3 models × 2 pipelines (baseline + enhanced) = **6 fits total** — typically 1–4 minutes.

To get the full academic results, run `Machine Learning.ipynb` overnight with the full grid.


In [ ]:
# ============================================================
# DATA LOADING — replicates Machine Learning.ipynb setup
# Run this cell ONCE before the quick-run cell below.
# ============================================================
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# --- Config (mirrors Machine Learning.ipynb) ---
class Config:
    dataset_dir    = r"Database\revenues_pseudonymized.xlsx"
    enrollees_dir  = r"Database\enrollees_pseudonymized.xlsx"
    parameters_dir = r"MachineLearning\parameters.json"
    cache_dir      = ""
    load_cache     = True
    observation_end = datetime.today()
    target_feature  = "dtp_bracket"
    test_size       = 0.3
    time_points     = [30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360, 390, 420, 450]

args = Config()

# --- Load raw tables ---
print("Loading revenues …")
df_revenues = pd.read_excel(args.dataset_dir)

print("Loading enrollees …")
df_enrollees = pd.read_excel(args.enrollees_dir)

# --- Build credit-sales ML table ---
from FeatureEngineering.credit_sales_machine_learning import CreditSales
print("Building credit sales features …")
cs = CreditSales(df_revenues, df_enrollees, args)
df_credit_sales = cs.show_data()

# --- Drop non-feature columns (mirrors Machine Learning.ipynb) ---
drop_columns = [
    'school_year', 'student_id_pseudonimized', 'category_name',
    'date_fully_paid', 'entry_date', 'due_date',
]
# Drop plan-type dummies that were excluded in the final run
plan_dummies_to_drop = [c for c in df_credit_sales.columns
                        if c.startswith('plan_type_Plan')]
drop_columns = drop_columns + plan_dummies_to_drop

# df_data  — fully-paid rows only (censor == 1), positive elapsed days
df_data = df_credit_sales[df_credit_sales['censor'] == 1].copy()
df_data = df_data[df_data['days_elapsed_until_fully_paid'] > 0]
df_data = df_data.drop(columns=[c for c in drop_columns if c in df_data.columns])

# df_data_surv — all rows (for survival feature generation), positive elapsed days
df_data_surv = df_credit_sales.copy()
df_data_surv = df_data_surv[df_data_surv['days_elapsed_until_fully_paid'] > 0]
df_data_surv = df_data_surv.drop(columns=[c for c in drop_columns if c in df_data_surv.columns])

# --- Best penalty (from final Coxnet search in Machine Learning.ipynb: 1e-06) ---
best_penalty = 1e-06

print(f"\n[OK] df_data      : {df_data.shape}")
print(f"[OK] df_data_surv : {df_data_surv.shape}")
print(f"[OK] best_penalty : {best_penalty}")
print(f"[OK] target_feature: {args.target_feature}")


In [15]:
# ============================================================
# QUICK RUN SETUP — ~1-4 minutes
# Produces MachineLearning/Results/model_results.xlsx
# ============================================================
import sys, os
from pathlib import Path
from types import SimpleNamespace

# Ensure project root is on sys.path
project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# --- Imports from project ---
from MachineLearning.Utils.run_models_parallel import SurvivalExperimentRunner
from MachineLearning.Models.decision_tree import DecisionTreePipeline
from MachineLearning.Models.gaussian_naive_bayes import GaussianNaiveBayesPipeline
from MachineLearning.Models.k_nearest_neighbor import KnearestNeighborPipeline

# ---------------------------------------------------------------
# 1. Load data — reuse df_data / df_data_surv already in scope
#    (loaded by Machine Learning.ipynb data-prep cells).
#    If those variables are not in scope, run those cells first.
# ---------------------------------------------------------------
try:
    _ = df_data
    _ = df_data_surv
    print("[OK] Using existing df_data / df_data_surv from kernel.")
except NameError:
    raise RuntimeError(
        "df_data / df_data_surv not found in kernel.\n"
        "Run the data-loading cells in Machine Learning.ipynb first "
        "(or copy them here), then re-run this cell."
    )

# ---------------------------------------------------------------
# 2. Quick-run configuration
# ---------------------------------------------------------------
QUICK_MODELS = {
    "gaussian_naive_bayes": GaussianNaiveBayesPipeline,
    "decision_tree":        DecisionTreePipeline,
    "knn":                  KnearestNeighborPipeline,
}

QUICK_STRATEGIES = ["borderline_smote"]   # 1 strategy → no repeated prep cost

# Shared args — must match what the full notebook uses
args = SimpleNamespace(
    parameters_dir  = "MachineLearning/parameters.json",
    target_feature  = "status",             # ← adjust if yours differs
    test_size       = 0.2,
    time_points     = [30, 60, 90, 180],    # ← adjust if yours differs
)

# best_penalty: reuse from kernel if available, else default to 0.1
try:
    _penalty = best_penalty
    print(f"[OK] Using best_penalty={_penalty} from kernel.")
except NameError:
    _penalty = 0.1
    print(f"[INFO] best_penalty not in scope — defaulting to {_penalty}.")

OUTPUT_PATH = "MachineLearning/Results/model_results.xlsx"

# ---------------------------------------------------------------
# 3. Restrict to FIRST param set only for each model
# ---------------------------------------------------------------
import json

with open(args.parameters_dir, encoding="utf-8") as f:
    full_params = json.load(f)

quick_params = {}
for model_key in QUICK_MODELS:
    if model_key in full_params and full_params[model_key]:
        quick_params[model_key] = [full_params[model_key][0]]   # first param only
    else:
        quick_params[model_key] = [{}]

QUICK_PARAMS_PATH = "MachineLearning/quick_params_temp.json"
with open(QUICK_PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump(quick_params, f, indent=2)

args.parameters_dir = QUICK_PARAMS_PATH
print(f"[OK] Quick params written to {QUICK_PARAMS_PATH}")
for m, v in quick_params.items():
    print(f"     {m}: {v}")

# ---------------------------------------------------------------
# 4. Run
# ---------------------------------------------------------------
print("\nStarting quick run …")
runner = SurvivalExperimentRunner(
    df_data                    = df_data,
    df_data_surv               = df_data_surv,
    models                     = QUICK_MODELS,
    balance_strategies         = QUICK_STRATEGIES,
    args                       = args,
    best_penalty               = _penalty,
    thresholds                 = None,
    n_jobs                     = -1,    # use all CPUs
    output_path                = OUTPUT_PATH,
    do_not_parallel_compute    = [],
    feature_selection_baseline = False, # OFF → biggest speed-up
    feature_selection_enhanced = False, # OFF → biggest speed-up
)

df_results = runner.run()

print(f"\n[DONE] {len(df_results)} result rows written to {OUTPUT_PATH}")
print(df_results[["model", "balance_strategy",
                   "baseline_f1_macro", "enhanced_f1_macro"]].to_string(index=False))

# Restore original parameters path for consistency
args.parameters_dir = "MachineLearning/parameters.json"


d:\Developer\Projects\Academic_Projects\THESIS-Utilizing-ML-to-Solve-the-IPPP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Developer\Projects\Academic_Projects\THESIS-Utilizing-ML-to-Solve-the-IPPP\.venv\Lib\site-packages\torch\cuda\__init__.py:184: UserWarning: cudaGetDeviceCount() returned cudaErrorNotSupported, likely using older driver or on CPU machine (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:88.)
  return torch._C._cuda_getDeviceCount() > 0


RuntimeError: df_data / df_data_surv not found in kernel.
Run the data-loading cells in Machine Learning.ipynb first (or copy them here), then re-run this cell.

In [ ]:

# Run the Dash app
# Open http://127.0.0.1:8050/ in your browser after running this cell.
import subprocess, sys

venv_python = r"d:\Developer\Projects\Academic_Projects\THESIS-Utilizing-ML-to-Solve-the-IPPP\.venv\Scripts\python.exe"
app_path    = r"d:\Developer\Projects\Academic_Projects\THESIS-Utilizing-ML-to-Solve-the-IPPP\app.py"

proc = subprocess.Popen([venv_python, app_path])
print(f"App started (PID {proc.pid}). Visit http://127.0.0.1:8050/")
print("To stop the app, run:  proc.terminate()")
